# 🏋️ Auto-Pilot Training — Surface Crack Detection

**Train all models sequentially, track with wandb, push to HF Hub.**

| Step | Model | Est. Time |
|------|-------|-----------|
| 5 | ResNet50 → warmup 5 + finetune 15 | ~3 min |
| 6 | EfficientNet-B0 | ~4 min |
| 7 | ViT-B/16 (included) | ~8 min |
| 8-9 | Ensemble evaluation | ~30 sec |
| 11 | Push to HF Hub | ~30 sec |

In [ ]:
# Cell 1: GPU check & mount Drive
import torch, sys, os, subprocess, time, json
from pathlib import Path

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

PROJECT_DIR = "/content/bootcamp-ace-26-team-7"
if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/anomalyco/bootcamp-ace-26-team-7.git "{PROJECT_DIR}"
%cd "{PROJECT_DIR}"

In [ ]:
# Cell 2: Install deps & wandb login
!pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu124
!pip install -q wandb

import wandb
from huggingface_hub import login as hf_login

# wandb login — prompts for API key
if not wandb.api.api_key:
    wandb.login()

print("Dependencies installed. wandb ready.")

In [ ]:
# Cell 3: Download dataset from HF Hub
HF_TOKEN = input("Enter HuggingFace token (write access, or press Enter to skip HF Hub): ").strip()

from huggingface_hub import hf_hub_download, HfApi

REPO_ID = "amruthjakku/surface-crack-detection-model"
RAW_DIR = "data/raw"

if not os.path.exists(RAW_DIR) or not os.listdir(RAW_DIR):
    print("Downloading dataset from HF Hub...")
    zip_path = hf_hub_download(repo_id=REPO_ID, filename="dataset.zip", repo_type="model")
    import zipfile
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("data")
    print(f"Dataset extracted to {RAW_DIR}/")
else:
    print(f"Dataset already exists at {RAW_DIR}/ — skipping download")

print("Raw images:", len(os.listdir(RAW_DIR)) if os.path.exists(RAW_DIR) else 0)

In [ ]:
# Cell 4: Prepare data (stratified split)
os.environ["WANDB_ENABLED"] = "true"
import src.config as cfg
cfg.Config.WANDB_ENABLED = True

from src.prepare_data import prepare_data
prepare_data()
print("Data prepared.")

In [ ]:
# Cell 5: Train ResNet50
print("=" * 60)
print("Training ResNet50...")
print("=" * 60)
from src.train import run_training
run_training(model_name="resnet50")
print("ResNet50 training complete.")

In [ ]:
# Cell 6: Train EfficientNet-B0
print("=" * 60)
print("Training EfficientNet-B0...")
print("=" * 60)
torch.cuda.empty_cache()
run_training(model_name="efficientnet_b0")
print("EfficientNet-B0 training complete.")

In [ ]:
# Cell 7: Train ViT-B/16
print("=" * 60)
print("Training ViT-B/16...")
print("=" * 60)
torch.cuda.empty_cache()
run_training(model_name="vit_b_16")
print("ViT-B/16 training complete.")

In [ ]:
# Cell 8: Evaluate each model individually
from src.evaluate import run_evaluation

results = {}
for model_name in ["resnet50", "efficientnet_b0", "vit_b_16"]:
    print(f"\n{'=' * 60}")
    print(f"Evaluating {model_name}...")
    print(f"{'=' * 60}")
    try:
        run_evaluation(model_name=model_name)
    except Exception as e:
        print(f"Failed to evaluate {model_name}: {e}")

In [ ]:
# Cell 9: Ensemble evaluation
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import Config
from src.dataset import get_dataloaders
from src.model import get_model

device = Config.DEVICE
models_to_load = Config.ENSEMBLE_MODELS

ensemble_nets = []
for name in models_to_load:
    path = Config.get_model_path(name)
    if not os.path.exists(path):
        print(f"Skipping {name} — checkpoint not found at {path}")
        continue
    net = get_model(model_name=name, num_classes=Config.NUM_CLASSES, pretrained=False)
    net.load_state_dict(torch.load(path, map_location=device))
    net = net.to(device)
    net.eval()
    ensemble_nets.append(net)
    print(f"Loaded {name} from {path}")

if len(ensemble_nets) < 2:
    print("Need at least 2 models for ensemble. Skipping.")
else:
    _, _, test_loader = get_dataloaders()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            probs = torch.zeros(images.size(0), Config.NUM_CLASSES).to(device)
            for net in ensemble_nets:
                logits = net(images)
                probs += torch.softmax(logits, dim=1)
            probs /= len(ensemble_nets)
            _, preds = torch.max(probs, 1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())

    y_true, y_pred = np.array(y_true), np.array(y_pred)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    report = classification_report(y_true, y_pred, target_names=Config.CLASSES)

    print(f"\n{'=' * 60}")
    print(f"Ensemble ({'+'.join(models_to_load)}) Results")
    print(f"{'=' * 60}")
    print(f"Test Accuracy: {acc:.4f}")
    print(f"Test Weighted F1: {f1:.4f}")
    print(f"\nClassification Report:\n{report}")

    # Log ensemble to wandb
    if Config.WANDB_ENABLED:
        import wandb
        ensemble_name = "ensemble-" + "+".join(models_to_load)
        wandb.init(project=Config.WANDB_PROJECT, entity=Config.WANDB_ENTITY, name=f"eval-{ensemble_name}-{time.strftime('%Y%m%d-%H%M%S')}")
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=Config.CLASSES, yticklabels=Config.CLASSES)
        plt.title(f"Confusion Matrix — {ensemble_name}")
        plt.ylabel("True Class")
        plt.xlabel("Predicted Class")
        cm_path = os.path.join(Config.REPORTS_DIR, f"confusion_matrix_{ensemble_name}.png")
        os.makedirs(Config.REPORTS_DIR, exist_ok=True)
        plt.savefig(cm_path); plt.close()
        wandb.log({"test_accuracy": acc, "test_weighted_f1": f1, "confusion_matrix": wandb.Image(cm_path)})
        wandb.finish()

In [ ]:
# Cell 10: Comparison summary
print("=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
# Load the printed results from logs — manual copy to README
print("\nRun wandb compare at: https://wandb.ai/<entity>/surface-crack-detection")
print("\nQuick links:")
print("  - ResNet50:       models/resnet50_best.pth")
print("  - EfficientNet-B0: models/efficientnet_b0_best.pth")
print("  - ViT-B/16:       models/vit_b_16_best.pth")
print(f"  - Ensemble:       {Config.ENSEMBLE_MODELS}")

In [ ]:
# Cell 11: Push best models to HF Hub (requires HF_TOKEN from Cell 3)
if HF_TOKEN:
    from huggingface_hub import HfApi, login
    login(token=HF_TOKEN)
    api = HfApi()

    for model_name in ["resnet50", "efficientnet_b0", "vit_b_16"]:
        path = Config.get_model_path(model_name)
        if os.path.exists(path):
            api.upload_file(
                path_or_fileobj=path,
                path_in_repo=f"{model_name}_best.pth",
                repo_id=REPO_ID,
                repo_type="model",
            )
            file_size_mb = os.path.getsize(path) / 1024 / 1024
            print(f"Pushed {model_name}_best.pth ({file_size_mb:.1f} MB) → {REPO_ID}")

    # Also push training curves
    for fname in os.listdir(Config.REPORTS_DIR):
        fpath = os.path.join(Config.REPORTS_DIR, fname)
        if os.path.isfile(fpath):
            api.upload_file(
                path_or_fileobj=fpath,
                path_in_repo=f"reports/{fname}",
                repo_id=REPO_ID,
                repo_type="model",
            )
            print(f"Pushed reports/{fname}")

    print(f"\n✅ All artifacts pushed to https://huggingface.co/{REPO_ID}")
else:
    print("No HF_TOKEN provided — skipping push.")

In [ ]:
# Cell 12: Open wandb dashboard
if wandb.api.api_key:
    dashboard_url = f"https://wandb.ai/{Config.WANDB_ENTITY or '<your-entity>'}/{Config.WANDB_PROJECT}"
    print(f"🔗 Open wandb dashboard: {dashboard_url}")
    from IPython.display import display, HTML
    display(HTML(f'<a href="{dashboard_url}" target="_blank">Open wandb Dashboard</a>'))
else:
    print("wandb not logged in. Run Cell 2 first.")

print("\n✅ Auto-pilot training complete!")